In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# Typical spam vs ham examples
spam_examples = [
    "URGENT! You've won $1000000! Click here NOW!",
    "FREE VIAGRA! No prescription needed! Call 555-SPAM",
    "Congratulations! You're our 1000th visitor! Claim your prize!",
    "Make $5000/week working from home! No experience required!",
    "STOP! Don't pay your credit card bill until you read this!"
]

ham_examples = [
    "Hey, are we still meeting for lunch tomorrow?",
    "Your package has been delivered to your front door.",
    "Meeting rescheduled to 3 PM. See you in conference room B.",
    "Thanks for the great presentation today!",
    "Your subscription renewal is due next month."
]


In [15]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
import torch

# 1. Prepare data
data = {
    'text': spam_examples + ham_examples,
    'labels': [1]*len(spam_examples) + [0]*len(ham_examples)  # 1=spam, 0=ham
}
dataset = Dataset.from_dict(data)

# 2. Load model
tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')
model = AutoModelForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=2)

# 2. Load model - Using Qwen2.5-0.5B
# tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen2.5-0.5B')
# model = AutoModelForSequenceClassification.from_pretrained('Qwen/Qwen2.5-0.5B', num_labels=2)


# 3. Tokenize
def tokenize(examples):
    return tokenizer(examples['text'], truncation=True, padding=True)

dataset = dataset.map(tokenize, batched=True)

# 4. Train (minimal setup)
training_args = TrainingArguments(
    output_dir='./spam_model',
    num_train_epochs=3,
    per_device_train_batch_size=8,
    logging_steps=10,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer
    # tokenizer=tokenizer,
)

trainer.train()  # Uncomment to train


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 11064.72it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Map: 100%|██████████| 10/10 [00:00<00:00, 3264.30 examples/s]


Step,Training Loss


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.51it/s]


TrainOutput(global_step=6, training_loss=0.6262750625610352, metrics={'train_runtime': 1.1688, 'train_samples_per_second': 25.667, 'train_steps_per_second': 5.133, 'total_flos': 131949947880.0, 'train_loss': 0.6262750625610352, 'epoch': 3.0})

In [16]:
# Predict on new text
test_texts = [
    "WIN FREE MONEY NOW!!!",
    "Let's grab coffee tomorrow"
]

device = model.device
inputs = tokenizer(test_texts, truncation=True, padding=True, return_tensors="pt").to(device)
outputs = model(**inputs)
predictions = torch.argmax(outputs.logits, dim=-1)

for text, pred in zip(test_texts, predictions):
    print(f"Text: {text}")
    print(f"Prediction: {'SPAM' if pred == 1 else 'HAM'}\n")


Text: WIN FREE MONEY NOW!!!
Prediction: SPAM

Text: Let's grab coffee tomorrow
Prediction: HAM



In [18]:
pred

tensor(0, device='cuda:0')